In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import plotly.express as px
import matplotlib.pyplot as plt
import re
from tqdm.notebook import tqdm
from collections import OrderedDict
from scipy.spatial import ConvexHull
from tools import simplexBasis, formula2string, element_content, compositionTranform, getEhull, encodeRGB, unique_elements

In [ ]:
neat_elements = pd.read_pickle("data/neat_elements-dft_subset.pkl.gz")
result = pd.read_pickle('data/train_ccs-gnn_predictions_dft_subset.pkl.gz')
simplex_basis = simplexBasis(len(unique_elements) - 1)
simplex_basis_df = pd.DataFrame(simplex_basis, columns=[f'SC{i}' for i in range(len(simplex_basis) - 1)])
simplex_basis_df['formation_energy_pa'] = 0
simplex_basis_df['formula'] = neat_elements.loc[list(unique_elements)]['formula'].values
simplex_basis_df['formula_str'] = simplex_basis_df['formula'].apply(formula2string)
simplex_basis_df['compound'] = unique_elements
simplex_basis_df['part'] = 'Neat elements'
simplex_basis_df['size'] = 6

# DFT-derived structures
print('Collecting DFT-derived structures...')
compositions = []
for i in tqdm(result.index):
    compositions += [[element_content(element, result.loc[i], relative=False) for element in unique_elements]]
compositions = np.array(compositions)
simplex_coords_dft = pd.DataFrame(list(map(lambda x: compositionTranform(x, simplex_basis), compositions)),
                                  columns=[f'SC{i}' for i in range(len(simplex_basis) - 1)])
simplex_coords_dft['formation_energy_pa'] = result['formation_energy_per_atom'].values
simplex_coords_dft['formula'] = result['formula'].values
simplex_coords_dft['formula_str'] = simplex_coords_dft['formula'].apply(formula2string)
simplex_coords_dft['compound'] = result['compound'].values
simplex_coords_dft['part'] = result['Space_group_no'].apply(lambda x: 'CCS (Pmm2 and higher)' if x > 8 else 'CCS (Cm and lower)').values
simplex_coords_dft['size'] = result['Space_group_no'].apply(lambda x: (4 if x > 8 else 1)).values

# 2DMD dataset
path_2dmd = 'data/2d-materials-point-defects-all-table.pkl.gz'
data_2dmd = pd.read_pickle(path_2dmd)
data_2dmd = data_2dmd[(data_2dmd['base'] == 'WSe2') | (data_2dmd['base'] == 'MoS2')].copy()
data_2dmd['natoms'] = data_2dmd['atomic_numbers'].apply(len)
data_2dmd['formation_energy_per_atom'] = data_2dmd['formation_energy'] / data_2dmd['natoms']
compositions = []
print('Collecting structures of the 2DMD dataset...')
for i in tqdm(data_2dmd.index):
    compositions += [[element_content(element, data_2dmd.loc[i], relative=False) for element in unique_elements]]
compositions = np.array(compositions)
simplex_coords_2dmd = pd.DataFrame(list(map(lambda x: compositionTranform(x, simplex_basis), compositions)),
                                   columns=[f'SC{i}' for i in range(len(simplex_basis) - 1)])
simplex_coords_2dmd['formation_energy_pa'] = data_2dmd['structure_formation_energy_per_atom'].values
simplex_coords_2dmd['formula'] = data_2dmd['formula'].values
simplex_coords_2dmd['formula_str'] = simplex_coords_2dmd['formula'].apply(formula2string)
simplex_coords_2dmd['compound'] = data_2dmd.index
simplex_coords_2dmd['part'] = '2DMD datasets'
simplex_coords_2dmd['size'] = .3

simplex_df = pd.concat((simplex_basis_df, simplex_coords_dft, simplex_coords_2dmd), axis=0)
simplex_df.set_index('compound', inplace=True, drop=True)

Plot training CSS and 2DMD structures within the three-dimensional compositional simplex defined by Mo, W, Se, S as pure constituents (vertices). 

In [ ]:
fig = px.scatter_3d(simplex_df, x='SC0', y='SC1', z='SC2', color='part', size='size', color_discrete_sequence=px.colors.qualitative.Bold,
                    hover_data=[simplex_df['part'], simplex_df['formation_energy_pa'], simplex_df.index, simplex_df['formula_str']])
fig.update_traces(marker=dict(line=dict(width=0,)), selector=dict(mode='markers'))

In [ ]:
camera = {"eye": {"x": 1.5, "y": 0.5, "z": 0.9}}
fig.update_layout(scene_camera=camera)
fig.show()

In [ ]:
available_energies_indexes = simplex_df.index
available_parts = simplex_df['part'].unique()
print(len(available_energies_indexes), 'energies available in ', ','.join(available_parts), 'parts')
points = simplex_df.iloc[:, :len(unique_elements)].values
hull = ConvexHull(points)
print(f'Convex hull dimensionality is {hull.ndim}D')

For training CCS entries plot pairwise relationships between the compositions, space groups, DFT-derived formation energies (eV/atom), energies above the convex hull (eV/atom), and number of defect positions in a 2×2×1 supercell.

In [ ]:
energies_df = pd.read_pickle('data/train_ccs-2dmd-convex_hull.pkl.gz')
result_energies_df = pd.merge(result, energies_df, how='left', left_on='compound', right_on=energies_df.index)
space_groups = dict(zip(result['Space_group_no'], result['Space_group_symbol']))
space_groups = OrderedDict(sorted(space_groups.items(), reverse=True))
sns.pairplot(result_energies_df[['W_nsites', 'U_nsites', 'Se_nsites', 'Np_nsites', 'Space_group_symbol', 
                                 'formation_energy_per_atom', 'E_above_hull', 'defect_nsites']], 
             hue='Space_group_symbol', corner=True,
             hue_order=[item[1] for item in space_groups.items() if item[0] in result['Space_group_no'].unique()], palette='crest')
plt.tight_layout()
plt.show()

Plot $E_{hull}$ distributions across configuration subsets of training CCS and 2DMD dataset (25 meV/atom bins). 

In [ ]:
plt.rcParams['axes.prop_cycle'] = plt.cycler(color=map(encodeRGB, px.colors.qualitative.Bold))
plt.figure(figsize=(6, 4))
plt.hist([1000 * energies_df[energies_df['part'] == part]['E_above_hull'] for part in available_parts], 
         bins=np.linspace(0, 1600, 65), stacked=True, label=available_parts, edgecolor='grey')
plt.xlabel('$E_{hull}$, meV/atom')
plt.ylabel('Number of structures')
plt.legend()
plt.grid(alpha=0.3)
plt.yscale('log')
plt.tight_layout()
plt.show()
plt.rcParams['axes.prop_cycle'] = plt.rcParamsDefault['axes.prop_cycle']

For 2DMD datasets with defect $MoS_2$ and $WSe_2$ monolayers in 8×8×1 supercells plot pairwise relationships between the properties and energetics. Among the data provided, DFT-derived HOMO-LUMO gaps, defect formation energies, $E_{DF}$ (formation_energy_per_atom, eV/atom), and number of defects are given. Additional properties – space groups and conventional formation energies, $E_F$, (structure_formation_energy, eV/atom) as recalculated using the 2DMD at a Glance code, and energies above the convex hull (eV/atom) are given for comparison.

In [ ]:
missed_data = [col for col in energies_df.columns if col not in data_2dmd.columns]
data_2dmd_energies = pd.concat((data_2dmd, energies_df[energies_df['part'] == '2DMD datasets'][missed_data]), axis=1)
sns.pairplot(data_2dmd_energies[['defect_nsites', 'base', 'structure_formation_energy_per_atom', 'formation_energy_per_atom']], 
             height=3, hue='base', palette='tab10', corner=True, plot_kws=dict(marker="o", linewidth=1, alpha=0.1))
plt.tight_layout()
plt.show()

print('Analysis of defect numbers remapping...')
print(data_2dmd['cell'].unique())
for dc in data_2dmd['defect_nsites'].unique():
    print(2 ** 2 * dc / 8 ** 2)

In [ ]:
sns.pairplot(data_2dmd_energies[['homo_lumo_gap', 'defect_nsites', 'formation_energy_per_atom', 'base', 
                                 'space_group_no', 'structure_formation_energy_per_atom', 'E_above_hull']], 
             hue='base', palette='tab10', corner=True, plot_kws=dict(marker="o", linewidth=1, alpha=0.1))
plt.tight_layout()
plt.show()

Plot space-group distributions of the structured low-defect (LDC) and sparse high-defect (HDC) 2DMD subsets versus number of defect positions.

In [ ]:
plt.figure(figsize=(9, 4))
numbers = data_2dmd.groupby(['defect_nsites', 'space_group_no']).size().to_frame()\
          .rename(mapper={0: 'Number_of_structures'}, axis=1).reset_index()\
          .pivot(columns='defect_nsites', index='space_group_no', values='Number_of_structures')
numbers.sort_index(ascending=False, inplace=True)
ccs_space_groups2add22dmd = numbers.index
numbers = pd.concat((numbers, pd.DataFrame(index=[sg for sg in ccs_space_groups2add22dmd if sg not in numbers.index])), axis=0)
numbers.sort_index(ascending=False, inplace=True)
numbers = numbers.T

ax = sns.heatmap(np.log10(numbers), cmap='YlOrBr', annot=numbers, linewidth=.1, 
                 cbar_kws={'label': 'lg (number of inequivalent structures)', 'drawedges': False, "shrink": 0.9}, fmt='g',
                 xticklabels=list(map(lambda x: f'{space_groups[x]}\n({x})', numbers.columns)))
ax.set(xlabel="Space group", ylabel="Number of defect positions")
ax.xaxis.tick_top()
plt.show()

For the 2DMD dataset plot $E_{hull}$ versus number of defect positions; the vertical line marks the LDC (≤3)/HDC (≥4) boundary.
Additionally for 2DMD datasets with defect $MoS_2$ and $WSe_2$ monolayers in 8×8×1 supercells plot enlarged figure of relationship between defect formation energies, $E_{DF}$ (formation_energy_per_atom, eV/atom), and energies above the convex hull (eV/atom).

In [ ]:
fig, axs = plt.subplots(nrows=1, ncols=2, figsize=(8, 4), sharey=True)
for base in data_2dmd_energies['base'].unique():
    mask = data_2dmd_energies['base'] == base
    sns.scatterplot(x=1000 * data_2dmd_energies[mask]['formation_energy_per_atom'], y=1000 * data_2dmd_energies[mask]['E_above_hull'], 
                    alpha=0.2, label=re.sub('2', '$_2$', base), ax=axs[1])
    sns.scatterplot(x=data_2dmd_energies[mask]['defect_nsites'], y=1000 * data_2dmd_energies[mask]['E_above_hull'], 
                    alpha=0.2, ax=axs[0])
axs[0].plot([3.5, 3.5], [0, 300], 'k', alpha=0.3, lw=1)
axs[0].text(3.5 - .5, 300, 'LDC', verticalalignment='top', horizontalalignment='right')
axs[0].text(3.5 + .5, 300, 'HDC', verticalalignment='top', horizontalalignment='left')
axs[1].plot([0, 300], [0, 300], 'k', alpha=0.3, lw=1, label='perfect\nmatch')
for i in range(2):
    axs[i].grid(alpha=0.3)
axs[1].legend(labelspacing=0.3)
axs[0].set_xlabel('Number of defect positions (2DMD)')
axs[1].set_xlabel('$E_{DF}$, meV/atom (2DMD)')
axs[0].set_ylabel('$E_{hull}$, meV/atom')
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(nrows=1, ncols=1, figsize=(4.25, 4), sharey=True)
for base in data_2dmd_energies['base'].unique():
    mask = data_2dmd_energies['base'] == base
    sns.scatterplot(x=data_2dmd_energies[mask]['defect_nsites'], y=1000 * data_2dmd_energies[mask]['E_above_hull'], 
                    alpha=0.2, ax=ax, label=re.sub('2', '$_2$', base))
ax.plot([3.5, 3.5], [0, 300], 'k', alpha=0.3, lw=1)
ax.text(3.5 - .5, 300, 'LDC', verticalalignment='top', horizontalalignment='right')
ax.text(3.5 + .5, 300, 'HDC', verticalalignment='top', horizontalalignment='left')
ax.grid(alpha=0.3)
ax.legend(title='Base monolayer', labelspacing=0.3)
ax.set_xlabel('Number of defect positions (out of 192 available)')
ax.set_ylabel('2DMD datasets: $E_{hull}$, meV/atom')
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(nrows=1, ncols=1, figsize=(4.25, 4), sharey=True)
for base in data_2dmd_energies['base'].unique():
    mask = data_2dmd_energies['base'] == base
    sns.scatterplot(x=1000 * data_2dmd_energies[mask]['formation_energy_per_atom'], y=1000 * data_2dmd_energies[mask]['E_above_hull'], 
                    alpha=0.2, label=re.sub('2', '$_2$', base), ax=ax)
ax.plot([0, 300], [0, 300], 'k', alpha=0.3, lw=1, label='perfect\nmatch')
ax.grid(alpha=0.3)
ax.legend(labelspacing=0.3)
ax.set_xlabel('$E_{DF}$, meV/atom (2DMD)')
ax.set_ylabel('$E_{hull}$, meV/atom')
plt.tight_layout()
plt.show()

Plot vicinity of the DFT four-dimensional convex hull, including structures below 25 meV/atom above the hull (fourth dimension as color). 

In [ ]:
df_energies_available = energies_df.dropna().copy()
df_energies_available['size'] = (3 - df_energies_available['E_above_hull'])
fig = px.scatter_3d(df_energies_available, x='SC0', y='SC1', z='SC2', color='E_above_hull', size='size', size_max=20, opacity=0.5,
                    hover_data=[df_energies_available['part'], df_energies_available.index, df_energies_available['formula_str']], 
                    color_continuous_scale='inferno')
fig.update_traces(marker=dict(line=dict(width=0)), selector=dict(mode='markers'))
fig.show()

In [ ]:
limit_Ehull = 25e-3
df_energies_on_hull = energies_df[(energies_df['E_above_hull'] < limit_Ehull) | (energies_df['part'] == '2DMD subsets')].copy()
df_energies_on_hull = df_energies_on_hull[df_energies_on_hull['E_above_hull'] < limit_Ehull]
df_energies_on_hull['size'] = 20 * (limit_Ehull - df_energies_on_hull['E_above_hull'])
fig = px.scatter_3d(df_energies_on_hull, x='SC0', y='SC1', z='SC2', color='E_above_hull', size='size', size_max=20, opacity=1,
                    hover_data=[df_energies_on_hull['part'], df_energies_on_hull['formation_energy_pa'],
                                df_energies_on_hull.index, df_energies_on_hull['formula_str']], 
                    color_continuous_scale='inferno')
fig.update_traces(marker=dict(line=dict(width=0)), selector=dict(mode='markers'))
fig.show()
fig.write_html("convex_hull.html")

In [ ]:
camera = dict(up=dict(x=-1, y=0, z=0), eye=dict(x=0, y=0, z=1.7))
fig.update_layout(scene_camera=camera)
fig.show()